## K-means on the Full Feature Sampled Dataset

#### Load Libraries

In [6]:
import pandas as pd
import os
import time
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from scipy.stats import mode
import numpy as np


os.chdir(r"C:\Users\ameli\Documents\GitHub\CS4630_Group3_Project2")

#### Trying K-means on full dataset

In [7]:
# Read file in 
df = pd.read_csv("data/raw/HIGGS.csv.gz", header = None)

In [8]:
df.columns = ['label',
                     'lepton_pT', 'lepton_eta', 'lepton_phi',
                     'missing_E_mag', 'missing_E_phi',
                     'jet1_pT', 'jet1_eta', 'jet1_phi', 'jet1_btag',
                     'jet2_pT', 'jet2_eta', 'jet2_phi', 'jet2_btag',
                     'jet3_pT', 'jet3_eta', 'jet3_phi', 'jet3_btag',
                     'jet4_pT', 'jet4_eta', 'jet4_phi', 'jet4_btag',
                     'm_jj', 'm_jjj', 'm_lv', 'm_jlv',
                     'm_bb', 'm_wbb', 'm_wwbb']

In [9]:
X = df.drop(columns="label")
y = df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [10]:
start_time = time.time()

kmeans = KMeans(n_clusters=2, init = 'k-means++', random_state = 42, n_init='auto') 

kmeans.fit(X_scaled)

end_time = time.time()

training_time = end_time - start_time
print(f"K-means runtime: {training_time:.2f} seconds")
print(f"Iterations: {kmeans.n_iter_}")

start_time = time.time()

y_kmeans = kmeans.predict(X_scaled)

end_time = time.time()
prediction_time = end_time - start_time

print(f"K-means prediction time: {prediction_time:.2f} seconds")

K-means runtime: 32.76 seconds
Iterations: 13
K-means prediction time: 3.99 seconds


In [11]:
mapped_labels = np.zeros_like(y_kmeans) # same length as y_kmeans

for i in range(2):
    bool_mask = (y_kmeans==i)
    mapped_labels[bool_mask] = mode(y[bool_mask], keepdims=True).mode[0]

accuracy = accuracy_score(y, mapped_labels)*100
print(f"The accuracy of the K-means algorithm on the full feature dataset is {accuracy}%.")


The accuracy of the K-means algorithm on the full feature dataset is 54.80231818181818%.


#### Add Results to Table

In [12]:
summary_path = "results/metrics/clustering_summary.csv"

df_summary = pd.read_csv(summary_path)

In [15]:
mask = (
    (df_summary["method"] == "KMeans Full 28") &
    (df_summary["dataset_version"] == "HIGGS.csv.gz(full)")
)

df_summary.loc[mask, "n_rows"] = X_scaled.shape[0]
df_summary.loc[mask, "n_features"] = X_scaled.shape[1]
df_summary.loc[mask, "training_time_seconds"] = training_time
df_summary.loc[mask, "prediction_time_seconds"] = prediction_time
df_summary.loc[mask, "iterations"] = kmeans.n_iter_
df_summary.loc[mask, "accuracy"] = accuracy

In [17]:
df_summary.to_csv(summary_path, index=False)

#### Read Sampled Data In

In [101]:
X = pd.read_csv("data/processed/X_sample.csv")
print("X Shape:", X.shape)
print("\nFirst rows of X:")
print(X.head())


y = pd.read_csv("data/processed/y_sample.csv")
print("y Shape:", y.shape)
print("\nFirst rows of y:")
print(y.head())

X Shape: (200000, 28)

First rows of X:
   lepton_pT  lepton_eta  lepton_phi  missing_E_mag  missing_E_phi   jet1_pT  \
0   0.548478   -0.058492    0.979747       0.886419       0.773973  1.551918   
1   0.577577   -0.320490    0.009849       0.824704      -1.251037  0.706566   
2   5.065509   -1.502888   -0.564538       1.623411      -0.701304  1.297067   
3   0.851907   -0.946752    0.742821       1.373478       1.176896  0.705192   
4   0.520844    0.889179   -1.101643       0.585859      -0.184083  1.157915   

   jet1_eta  jet1_phi  jet1_btag   jet2_pT  ...  jet4_eta  jet4_phi  \
0  0.937722  0.576157   2.173076  1.391360  ...  2.452204 -1.606737   
1 -1.638856 -0.636922   0.000000  1.251607  ... -1.203890  1.031420   
2 -2.412226  0.922641   0.000000  2.297109  ... -1.362127 -1.156149   
3  0.044535 -0.453421   2.173076  0.585955  ... -1.152255 -1.049051   
4  0.453500  0.072784   0.000000  0.910912  ...  1.025577 -1.585096   

   jet4_btag      m_jj     m_jjj      m_lv     m_jlv

#### Scale Features

In [102]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

#### Fit Model and Predict

In [103]:
start_time = time.time()

kmeans = KMeans(n_clusters=2, init = 'k-means++', random_state = 42, n_init=20) 

kmeans.fit(X_scaled)

end_time = time.time()

training_time = end_time - start_time
print(f"K-means runtime: {training_time:.2f} seconds")
print(f"Iterations: {kmeans.n_iter_}")

start_time = time.time()

y_kmeans = kmeans.predict(X_scaled)

end_time = time.time()
prediction_time = end_time - start_time

print(f"K-means prediction time: {prediction_time:.2f} seconds")

K-means runtime: 1.63 seconds
Iterations: 6
K-means prediction time: 0.02 seconds


Parameters:
- init = 'k-means++' provides a more robust starting point. Instead of a random centroid initialization, k-means++ first chooses a random centroid from the data points. Then for each subsequent centorid, it selects a data point with a probability proportional to its squared distance from the nearest existing centroid. This process favors points that are further away from chosen centers, leading to a more dispersed and strategic intital placement. It increases the likelihood of finding a better final clustering solution and often reduces the number of iterations needed for convergence. 

- n_clusters=2 since there are two distinct classes, signal and background.

- random_state = 42 for reproducibility.

- n_init=20 for more stability

- max_iter = 300 is the default; convergence happens much earlier (~6 iterations)

#### Evaluate with y labels

Since K-means is unsupervised, the clusters might not correspond with the true label. Since we have the label, the following code below maps the cluster labels to the majority true label inside each cluster. 

In [104]:
mapped_labels = np.zeros_like(y_kmeans) # same length as y_kmeans

for i in range(2):
    bool_mask = (y_kmeans==i)
    mapped_labels[bool_mask] = mode(y[bool_mask], keepdims=True).mode[0]

accuracy = accuracy_score(y, mapped_labels)*100
print(f"The accuracy of the K-means algorithm on the full feature sampled dataset is {accuracy}%.")


The accuracy of the K-means algorithm on the full feature sampled dataset is 55.6385%.


The K-means clustering achieved an accuracy of ~ 55.64% when the cluster labels were mapped to the majority class within each cluster. This is slightly higher than the baseline accuracy of 54.33%, which was calculated in the `prepare_data.ipynb`. This indicates the clusters that the K-means algorithm formed do not align well with true signal/background labels. This makes sense, as K-means groups observations based on feature similarity rather than class labels. The full feature dataset contains a lot of similar features that overlap, so the accuracy and other metrics should be improved after PCA. 

#### Add Results to clustering_summary.csv

In [ ]:
# Add to results table to compare models
new_row = pd.DataFrame([{
    "method": "KMeans Full 28",
    "dataset_version": "sample_200k",
    "n_clusters": 2,
    "n_rows": X_scaled.shape[0],
    "n_features": X_scaled.shape[1],
    "training_time_seconds": training_time,
    "prediction_time_seconds": prediction_time,
    "iterations": kmeans.n_iter_,
    "accuracy": accuracy,
    "silhouette_score": None,
    "davies_bouldin": None,
    "compactness": None,
    "separation": None
}])


df_summary = pd.concat([df_summary, new_row], ignore_index=True)

df_summary.to_csv(summary_path, index=False)

           method     dataset_version n_clusters    n_rows n_features  \
0  KMeans Full 28  HIGGS.csv.gz(full)          2  10721302         28   
1  KMeans Full 28         sample_200k          2    200000         28   

   training_time_seconds  prediction_time_seconds iterations  accuracy  \
0              17.719350                 2.456887         18  55.68153   
1               1.632142                 0.024272          6  55.63850   

  silhouette_score davies_bouldin compactness separation  
0             None           None        None       None  
1             None           None        None       None  
